In [2]:
# load libraries 
import pandas as pd 
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pypq
import pyarrow.csv as pycsv
import textwrap 
from time import time 
import plotly.io as pio
import os
import networkx as nx


tqdm.pandas()
plt.rcParams.update({'font.size': 22})
sns.set(style="ticks", context="talk")
plt.style.use("dark_background")
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark+presentation'

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [3]:
# some helper functions 
def read_parquet(path, engine='pyarrow', columns=None, convert_dtypes=True, **args):
    """
    Read a parquet file (or a directory of parquet files) 
    columns: list of columns to read, by default, read all columns
    convert_dtypes: if True, convert datatypes to save RAM (takes extra time)
    """
    
    path = Path(path)
    name = path.stem 
    column_st = 'columns="all"' if columns is None else f'{columns=!r}'
    print(f'\nReading {column_st} from {path!r} using {engine=!r}.')

    tic = time()
    df = pd.read_parquet(path, engine=engine, columns=columns, **args)
    toc = time()
    print(f'Read {len(df):,} rows from {path.stem!r} in {toc-tic:.2f} sec.')
    
    if convert_dtypes:
        tic = time()
        size_before = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024

        string_cols_d = {}
        for col, dtype in df.dtypes.to_dict().items():
            if dtype == 'object':  # convert object columns to string
                string_cols_d[col] = 'string[python]'
            if col == 'type' or col == 'concept_name':
                if dtype != 'category':
                    string_cols_d[col] = 'category'
            if col == 'publication_month':
                if dtype != 'uint8':
                    string_cols_d[col] = 'uint8'
            if col == 'score':
                if dtype != 'float16':
                    string_cols_d[col] = 'float16'
        # print(f'{string_cols_d=}')
        df = df.astype(string_cols_d) 
        
        size_after = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024
        toc = time()
        print(f'Converting dtypes took {toc-tic:.2f} sec. Size before: {size_before:.2f}GB, after: {size_after:.2f}GB.')
    
    display('Top 3 rows:', df.head(3))
    return df


def peek_parquet(path):
    """
    peeks at a parquet file (or a directory containing parquet files) without reading the whole thing and prints the following:
    * Path
    * schema
    * number of pieces (fragments)
    * number of rows 
    """

    path = Path(path)
    parq_file = pypq.ParquetDataset(path)
    piece_count = len(parq_file.fragments)
    schema = textwrap.indent(parq_file.schema.to_string(), ' '*4)
    row_count = sum(frag.count_rows() for frag in parq_file.fragments)
    if Path(path).is_dir():
      size = sum(Path(frag.path).stat().st_size for frag in parq_file.fragments)
    else:
      size = path.stat().st_size
    
    st = [
        f'Name: {path.stem!r}',  
        f'Path: {str(path)!r}',
        f'Size: {size/1024/1024/1024:.2g} GB',
        f'Files: {piece_count:,}',
        f'Rows: {row_count:,}',
        f'Schema:\n{schema}',
        f'5 random rows:',
    ]
    print('\n'.join(st))
    sample_df = parq_file.fragments[0].head(5).to_pandas()  # read 5 rows from the first fragment
    display(sample_df)

    return

def read_smaller_tables(name):
    """
    Some smaller tables exist as a CSV only
    """
    assert name in ['institutions', 'institutions_geo', 'concepts']
    path = basepath / 'csv-files'/ month / name
    df = pd.read_csv(f'{path}.csv.gz', engine='c')
    return df

'''def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, chunksize=100000, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    else: 
        output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        
        chunks = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False,
            chunksize=chunksize,
            **read_csv_kwargs
        )

        for i, chunk in enumerate(chunks):
            chunk.to_parquet(
                output_path,
                index=False,
                engine='fastparquet',
                append=(i != 0)
            )
            
            if (i + 1) % 10 == 0:
                print(f"Processed {(i + 1) * chunksize} rows...")


        print(f"Successfully wrote Parquet file to: {output_path}")

    return output_path  '''

def tsv_to_parquet_pyarrow(tsv_path, output_filename=None, column_types=None, drop_columns=None):
    """
    Convert a TSV file to Parquet format using PyArrow directly.
    
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        column_types: Optional dict of column types (e.g., {'patent_id': pa.string()})
    
    Returns:
        Path to the created parquet file
    """
    notebook_dir = Path.cwd() / '..' / 'parquets'
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename
    
    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        
        # Set up parse options for TSV
        parse_options = pycsv.ParseOptions(delimiter='\t')

        # Figure out which columns to include
        include_columns = None
        if drop_columns:
            peek = pycsv.open_csv(tsv_path, parse_options=parse_options)
            first_batch = next(iter(peek))
            include_columns = [c for c in first_batch.schema.names if c not in drop_columns]
            
        # Set up convert options with column types if provided
        convert_options = None
        if column_types or include_columns:
            convert_options = pycsv.ConvertOptions(column_types=column_types, 
                                                   timestamp_parsers=["%Y-%m-%d"],
                                                   include_columns=include_columns)
        
        # Read TSV directly into Arrow table (streams data, doesn't load all into memory)
        reader = pycsv.open_csv(
            tsv_path,
            parse_options=parse_options,
            convert_options=convert_options
        )

        writer = None
        for batch in reader:
            if writer is None:
                writer = pypq.ParquetWriter(output_path, batch.schema)
            writer.write_batch(batch)
        if writer:
            writer.close()
             
    
    return output_path

def split_tsv_sequential(input_file, num_splits=5):
    """Split TSV into 5 sequential chunks with _1, _2, _3, _4, _5 naming"""
    
    # Get base filename without extension
    base_name = input_file.replace('.tsv', '')
    
    # First pass: count total rows
    print("Counting rows...")
    with open(input_file, 'r', encoding='utf-8') as f:
        header = f.readline()
        total_rows = sum(1 for _ in f)
    
    print(f"Total rows: {total_rows:,}")
    
    rows_per_file = total_rows // num_splits
    print(f"Rows per file: ~{rows_per_file:,}")
    
    # Second pass: split the file
    with open(input_file, 'r', encoding='utf-8') as f:
        header = f.readline()
        
        file_num = 1
        row_count = 0
        output = open(f'{base_name}_{file_num}.tsv', 'w', encoding='utf-8')
        output.write(header)  # Write header to first file
        
        for line in f:
            output.write(line)
            row_count += 1
            
            # Start new file when we hit the threshold
            if row_count >= rows_per_file and file_num < num_splits:
                output.close()
                print(f"Created {base_name}_{file_num}.tsv with {row_count:,} rows")
                
                file_num += 1
                row_count = 0
                output = open(f'{base_name}_{file_num}.tsv', 'w', encoding='utf-8')
                output.write(header)  # Write header to each file
        
        output.close()
        print(f"Created {base_name}_{file_num}.tsv with {row_count:,} rows")
        print("\nDone! Created files:")
        for i in range(1, num_splits + 1):
            print(f"  - {base_name}_{i}.tsv")

# Usage - replace 'your_file.tsv' with your actual filename


In [ ]:
USPTOPath = '/data/shared/USPTO-patents/'

##tsv_files = [f for f in os.listdir(USPTOPath) if f.endswith('.tsv')]
tsv_files = list(Path(USPTOPath).glob('*.tsv'))

# Convert all TSV files (function will skip if parquet already exists)
for tsv_file in tsv_files:
    ##tsv_file = Path(tsv_file)
    print(tsv_file.stem)
    if tsv_file.stem in ['g_us_application_citation', 'g_us_patent_citation']:
        continue
   """ tsv_to_parquet_pyarrow(
        tsv_file,
        column_types={'patent_id': pa.string(), 'patent_date': pa.timestamp("s")} #pa.dictionary() takes 2 values takes the key string and the second part would be the number of unique values there are (make numeric mapping based on each unique category
    ) """
# we cant use the same one for all tsvs as they all have different things  

print("All conversions complete!")

In [4]:
USPTOPath = '/data/shared/USPTO-patents/'

cpc_df = pd.read_csv(USPTOPath+'g_cpc_title.tsv', sep = '\t')
cpc_df


,cpc_subclass,cpc_subclass_title,cpc_group,cpc_group_title,cpc_class,cpc_class_title
0,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/00,Hand tools,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/02,Hand tools -Spades; Shovels,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
2,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/022,Hand tools -Spades; Shovels -Collapsible; exte...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
3,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/024,Hand tools -Spades; Shovels -Foot protectors a...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
4,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/026,Hand tools -Spades; Shovels -with auxiliary ha...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
...,...,...,...,...,...,...
269512,Y10T,TECHNICAL SUBJECTS COVERED BY FORMER US CLASSI...,Y10T83/9498,Cutting-Tool or tool with support-Stationary c...,Y10,TECHNICAL SUBJECTS COVERED BY FORMER USPC
269513,Y10T,TECHNICAL SUBJECTS COVERED BY FORMER US CLASSI...,Y10T83/95,Cutting-Machine frame,Y10,TECHNICAL SUBJECTS COVERED BY FORMER USPC
269514,Y10T,TECHNICAL SUBJECTS COVERED BY FORMER US CLASSI...,Y10T83/96,Cutting-Machine frame-Guard,Y10,TECHNICAL SUBJECTS COVERED BY FORMER USPC
269515,Y10T,TECHNICAL SUBJECTS COVERED BY FORMER US CLASSI...,Y10T83/97,Cutting-Miscellaneous,Y10,TECHNICAL SUBJECTS COVERED BY FORMER USPC


In [5]:
cpc_df[['cpc_subclass', 'cpc_group', 'cpc_class']].nunique()

cpc_subclass       681
cpc_group       269516
cpc_class          137
dtype: int64

In [10]:
'''tsv_to_parquet_pyarrow(
    USPTOPath+'g_cpc_title.tsv', 
    column_types={'cpc_subclass': pa.dictionary(pa.int32(), pa.string()), 'cpc_group': pa.string(), 'cpc_class': pa.dictionary(pa.int32(), pa.string())}
)
# key mapped to a value
'''

Converting /data/shared/USPTO-patents/g_cpc_title.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/g_cpc_title.parquet')

In [11]:
df = pd.read_parquet('g_cpc_title.parquet')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 269517 entries, 0 to 269516
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype   
---  ------              --------------   -----   
 0   cpc_subclass        269517 non-null  category
 1   cpc_subclass_title  269517 non-null  object  
 2   cpc_group           269517 non-null  object  
 3   cpc_group_title     269517 non-null  object  
 4   cpc_class           269517 non-null  category
 5   cpc_class_title     269517 non-null  object  
dtypes: category(2), object(4)
memory usage: 9.3+ MB


In [12]:
peek_parquet('g_cpc_title.parquet')

Name: 'g_cpc_title'
Path: 'g_cpc_title.parquet'
Size: 0.011 GB
Files: 1
Rows: 269,517
Schema:
    cpc_subclass: dictionary<values=string, indices=int32, ordered=0>
    cpc_subclass_title: string
    cpc_group: string
    cpc_group_title: string
    cpc_class: dictionary<values=string, indices=int32, ordered=0>
    cpc_class_title: string
5 random rows:


,cpc_subclass,cpc_subclass_title,cpc_group,cpc_group_title,cpc_class,cpc_class_title
0,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/00,Hand tools,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/02,Hand tools -Spades; Shovels,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
2,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/022,Hand tools -Spades; Shovels -Collapsible; exte...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
3,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/024,Hand tools -Spades; Shovels -Foot protectors a...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
4,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/026,Hand tools -Spades; Shovels -with auxiliary ha...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...


In [ ]:
#how will sql handle categorical columns (TODO research on this)

In [4]:
USPTOPath = '/data/shared/USPTO-patents/'

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_1.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

Converting /data/shared/USPTO-patents/g_us_patent_citation_1.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/g_us_patent_citation_1.parquet')

In [7]:
peek_parquet('../parquets/g_us_patent_citation_1.parquet')

Name: 'g_us_patent_citation_1'
Path: '../parquets/g_us_patent_citation_1.parquet'
Size: 0.39 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,10000000,0,5093563,1992-03-01,Small,cited by examiner
1,10000000,1,5751830,1998-05-01,Hutchinson,cited by applicant
2,10000001,0,7804268,2010-09-01,Park,cited by examiner
3,10000001,1,9022767,2015-05-01,Oono,cited by examiner
4,10000001,2,9090016,2015-07-01,Takeuchi,cited by examiner


In [8]:
USPTOPath = '/data/shared/USPTO-patents/'

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_2.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_3.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_4.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_5.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_us_patent_citation_6.tsv',
                       column_types={'patent_id': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
                       drop_columns=['wipo_kind']
                      )

Converting /data/shared/USPTO-patents/g_us_patent_citation_2.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_us_patent_citation_3.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_us_patent_citation_4.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_us_patent_citation_5.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_us_patent_citation_6.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/g_us_patent_citation_6.parquet')

In [12]:
USPTOPath = '/data/shared/USPTO-patents/'

for i in range(1,6):
    tsv_to_parquet_pyarrow(
        USPTOPath + f'g_us_application_citation_{i}.tsv',
        column_types={'patent_id': pa.string(), 'citation_document_number': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
        drop_columns=['wipo_kind']
    )
        

Converting /data/shared/USPTO-patents/g_us_application_citation_5.tsv to Parquet...


In [13]:
peek_parquet('../parquets/g_us_application_citation_3.parquet')

Name: 'g_us_application_citation_3'
Path: '../parquets/g_us_application_citation_3.parquet'
Size: 0.42 GB
Files: 1
Rows: 15,434,436
Schema:
    patent_id: string
    citation_sequence: int64
    citation_document_number: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_document_number,citation_date,record_name,citation_category
0,6963874,9,20030023875,2003-01-01,Ouchi et al.,cited by other
1,9704331,8,20050124411,2005-06-01,Schneider et al.,cited by applicant
2,10808433,19,20150284977,2015-10-01,Barmscheidt,cited by examiner
3,11710299,49,20090040215,2009-02-01,Afzulpurkar et al.,cited by applicant
4,7737724,29,20030061572,2003-03-01,McClannahan et al.,cited by other


In [5]:
USPTOPath = '/data/shared/USPTO-patents/'
tsv_to_parquet_pyarrow(
        USPTOPath + 'g_us_application_citation_1.tsv',
        column_types={'patent_id': pa.string(), 'citation_document_number': pa.string(), 'citation_date': pa.timestamp("s"), 'citation_category': pa.dictionary(pa.int32(), pa.string())},
        drop_columns=['wipo_kind']
    )

Converting /data/shared/USPTO-patents/g_us_application_citation_1.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/../parquets/g_us_application_citation_1.parquet')

In [7]:
peek_parquet('../parquets/g_us_patent_citation_1.parquet')


Name: 'g_us_patent_citation_1'
Path: '../parquets/g_us_patent_citation_1.parquet'
Size: 0.39 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,10000000,0,5093563,1992-03-01,Small,cited by examiner
1,10000000,1,5751830,1998-05-01,Hutchinson,cited by applicant
2,10000001,0,7804268,2010-09-01,Park,cited by examiner
3,10000001,1,9022767,2015-05-01,Oono,cited by examiner
4,10000001,2,9090016,2015-07-01,Takeuchi,cited by examiner


In [8]:
peek_parquet('../parquets/g_us_patent_citation_2.parquet')
peek_parquet('../parquets/g_us_patent_citation_3.parquet')
peek_parquet('../parquets/g_us_patent_citation_4.parquet')
peek_parquet('../parquets/g_us_patent_citation_5.parquet')
peek_parquet('../parquets/g_us_patent_citation_6.parquet')

Name: 'g_us_patent_citation_2'
Path: '../parquets/g_us_patent_citation_2.parquet'
Size: 0.4 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,11278279,1487,6123241,2000-09-01,Walter et al.,cited by applicant
1,11278279,1488,6123701,2000-09-01,Nezhat,cited by applicant
2,11278279,1489,H1904,2000-10-01,Yates et al.,cited by applicant
3,11278279,1490,RE36923,2000-10-01,Hiroi et al.,cited by applicant
4,11278279,1491,6126058,2000-10-01,Adams et al.,cited by applicant


Name: 'g_us_patent_citation_3'
Path: '../parquets/g_us_patent_citation_3.parquet'
Size: 0.43 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,4446740,0,3815409,1974-06-01,Macovski,
1,4446740,1,4016750,1977-04-01,Green,
2,4446740,2,4290310,1981-09-01,Anderson,
3,4446740,3,4307613,1981-12-01,Fox,
4,4446740,4,4319489,1982-03-01,Yamaguchi et al.,


Name: 'g_us_patent_citation_4'
Path: '../parquets/g_us_patent_citation_4.parquet'
Size: 0.41 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,6919172,17,5827741,1998-10-01,Beattie et al.,cited by other
1,6919172,18,6468782,2002-10-01,Tunnacliffe et al.,cited by other
2,6919173,0,6689772,2004-02-01,Boschelli et al.,cited by examiner
3,6919174,0,3413464,1968-11-01,Kamentsky,cited by other
4,6919174,1,4101279,1978-07-01,Aslam,cited by other


Name: 'g_us_patent_citation_5'
Path: '../parquets/g_us_patent_citation_5.parquet'
Size: 0.4 GB
Files: 1
Rows: 25,190,121
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,8307542,37,6452735,2002-09-01,Egan et al.,cited by other
1,8307542,38,6452755,2002-09-01,Bonin,cited by other
2,8307542,39,6465929,2002-10-01,Levitan et al.,cited by other
3,8307542,40,6469859,2002-10-01,Chainer et al.,cited by other
4,8307542,41,6487045,2002-11-01,Yanagisawa,cited by other


Name: 'g_us_patent_citation_6'
Path: '../parquets/g_us_patent_citation_6.parquet'
Size: 0.37 GB
Files: 1
Rows: 25,190,124
Schema:
    patent_id: string
    citation_sequence: int64
    citation_patent_id: string
    citation_date: timestamp[ms]
    record_name: string
    citation_category: dictionary<values=string, indices=int32, ordered=0>
5 random rows:


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category
0,9603099,9,8228855,2012-07-01,Sambhwani et al.,cited by applicant
1,9603099,10,8315320,2012-11-01,Zhang et al.,cited by applicant
2,9603099,11,8335466,2012-12-01,Cai et al.,cited by applicant
3,9603099,12,8355388,2013-01-01,Womack et al.,cited by applicant
4,9603099,13,8402334,2013-03-01,Yu et al.,cited by applicant


In [6]:
USPTOPath = '/data/shared/USPTO-patents/'

tsv_to_parquet_pyarrow(USPTOPath+'g_cpc_current.tsv',
                       column_types={'patent_id': pa.string(), 'cpc_subclass': pa.dictionary(pa.int32(), pa.string()), 'cpc_group': pa.string(), 'cpc_class': pa.dictionary(pa.int32(), pa.string()), 'cpc_section': pa.dictionary(pa.int32(), pa.string())}
                      )


Converting /data/shared/USPTO-patents/g_cpc_current.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/../parquets/g_cpc_current.parquet')

In [8]:
peek_parquet('../parquets/g_cpc_current.parquet')

Name: 'g_cpc_current'
Path: '../parquets/g_cpc_current.parquet'
Size: 0.66 GB
Files: 1
Rows: 57,969,447
Schema:
    patent_id: string
    cpc_sequence: int64
    cpc_section: dictionary<values=string, indices=int32, ordered=0>
    cpc_class: dictionary<values=string, indices=int32, ordered=0>
    cpc_subclass: dictionary<values=string, indices=int32, ordered=0>
    cpc_group: string
    cpc_type: string
5 random rows:


,patent_id,cpc_sequence,cpc_section,cpc_class,cpc_subclass,cpc_group,cpc_type
0,3950000,0,A,A63,A63C,A63C9/001,inventional
1,3950000,1,A,A63,A63C,A63C9/00,inventional
2,3950000,2,A,A63,A63C,A63C9/002,inventional
3,3950000,3,A,A63,A63C,A63C9/081,inventional
4,3950001,0,A,A63,A63C,A63C9/086,inventional


In [9]:
USPTOPath = '/data/shared/USPTO-patents/'
split_tsv_sequential(USPTOPath+'g_cpc_current.tsv', 3)

Counting rows...
Total rows: 57,969,447
Rows per file: ~19,323,149
Created /data/shared/USPTO-patents/g_cpc_current_1.tsv with 19,323,149 rows
Created /data/shared/USPTO-patents/g_cpc_current_2.tsv with 19,323,149 rows
Created /data/shared/USPTO-patents/g_cpc_current_3.tsv with 19,323,149 rows

Done! Created files:
  - /data/shared/USPTO-patents/g_cpc_current_1.tsv
  - /data/shared/USPTO-patents/g_cpc_current_2.tsv
  - /data/shared/USPTO-patents/g_cpc_current_3.tsv


In [11]:
USPTOPath = '/data/shared/USPTO-patents/'

tsv_to_parquet_pyarrow(USPTOPath+'g_cpc_current_1.tsv',
                       column_types={'patent_id': pa.string(), 'cpc_subclass': pa.dictionary(pa.int32(), pa.string()), 'cpc_group': pa.string(), 'cpc_class': pa.dictionary(pa.int32(), pa.string()), 'cpc_section': pa.dictionary(pa.int32(), pa.string())}
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_cpc_current_2.tsv',
                       column_types={'patent_id': pa.string(), 'cpc_subclass': pa.dictionary(pa.int32(), pa.string()), 'cpc_group': pa.string(), 'cpc_class': pa.dictionary(pa.int32(), pa.string()), 'cpc_section': pa.dictionary(pa.int32(), pa.string())}
                      )

tsv_to_parquet_pyarrow(USPTOPath+'g_cpc_current_3.tsv',
                       column_types={'patent_id': pa.string(), 'cpc_subclass': pa.dictionary(pa.int32(), pa.string()), 'cpc_group': pa.string(), 'cpc_class': pa.dictionary(pa.int32(), pa.string()), 'cpc_section': pa.dictionary(pa.int32(), pa.string())}
                      )


Converting /data/shared/USPTO-patents/g_cpc_current_1.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_cpc_current_2.tsv to Parquet...
Converting /data/shared/USPTO-patents/g_cpc_current_3.tsv to Parquet...


PosixPath('/home/jupyter-mgarciamelo/ipynb/../parquets/g_cpc_current_3.parquet')